<a href="https://colab.research.google.com/github/TanWaiKen/RL-workshop/blob/main/tool_call_rl.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Workshop Scope
- **Goal:** Train `Qwen/Qwen2.5-1.5B-Instruct` using RL (GRPO, DPO, RLOO) to select valid Relationship Simulator partner tools under Colab memory limits.
- **Focus:** Verifiable tool-call formatting (`<think>`, `<tool_call>`), argument correctness, and love-meter outcomes.
- **Data:** 100% environment-grounded simulator scenarios (two scenario sets: training & held-out eval, no external tool datasets).


# 💞 Tool-Calling RL with TRL — Relationship Simulator

- **Model:** `Qwen/Qwen2.5-1.5B-Instruct` with LoRA
- **Target Hardware:** Free Google Colab T4 GPU (16 GB, fp16) or L4 (24 GB, bf16)
- **Stack:** TRL 1.12.0, transformers 4.57.6, peft 0.20.0, accelerate 1.14.0, datasets 4.8.5, gradio

### Pipeline
1. **Setup:** Install pinned stack & verify GPU.
2. **Data:** Generate two sets of Relationship Simulator scenarios (Train vs Held-out Eval).
3. **SFT:** Warm up model on optimal tool-calling demonstrations.
4. **GRPO:** Group Relative Policy Optimization (primary RL method).
5. **DPO:** Direct Preference Optimization (comparison method on chosen vs rejected pairs).
6. **RLOO:** Reinforce Leave-One-Out (comparison method with sibling baseline).
7. **Benchmark:** Compare Base, SFT, GRPO, DPO, and RLOO on BFCL categories & love gain.
8. **Interactive Demo:** In-notebook Gradio chat to test models live.


## Why RL for Tool Calling?
- **SFT limitation:** SFT only imitates fixed demonstration text.
- **RL advantage:** The model explores actions and learns from verifiable simulator outcomes.
- **Deterministic reward:** The simulator calculates objective love-meter point changes for each tool call.
- **Fair comparison:** GRPO, DPO, and RLOO all start from the same SFT checkpoint and share the same prompt budget.


## Model Selection: Qwen2.5-1.5B-Instruct
- **VRAM Footprint:** ~9 GB on T4 (15 GB usable) with LoRA `r=16`, gradient checkpointing, and 4 completions.
- **Format Adherence:** 1.5B provides reliable capacity to follow JSON tool-call schema and argument types.
- **Fallback Support:** If Hugging Face is down or rate-limited, the model is pulled from `Qwen2.5-1.5B-Instruct.zip` on Google Drive.


## 🎮 Simulator Rules & Tools
- **Partner State:** Mood (`happy`, `sad`, `tired`, `stressed`, `angry`), Love Meter (0–100), Occasion, Preferences.
- **Action Budget:** Up to 3 tool calls per response (extra calls penalized).
- **The 9 Partner Tools:**
  - `give_gift(item)`: Give gift matching partner's liked types.
  - `watch_movie(genre)`: Watch a movie genre they like.
  - `cook_dinner(dish)`: Cook favorite dish.
  - `hug()`, `kiss()`: Physical affection.
  - `apologize(reason)`: Mandatory after an argument before other actions work.
  - `plan_date(activity)`: Plan activities (e.g. stargazing).
  - `send_message(text)`: Reassuring message.
  - `ask_how_partner_feels()`: Discover partner's mood.
- **Outcome Scoring:** Exact matching of preferences, penalty for disliked genres or missing apologies.


## Step 1: Install Dependencies & Setup Colab
- **Colab Setup (Optional):** If using GitHub clone, run:
  ```bash
  # !git clone <repo_url>
  # %cd <repo_dir>
  ```
- **Stack:** `trl==1.12.0`, `transformers==4.57.6`, `peft==0.20.0`, `accelerate==1.14.0`, `datasets==4.8.5`, `gradio`.
- **Note:** Uninstalls `torchao` to avoid peft conflict. Restart session after running if prompted.


In [ ]:
# Pinned stack + gradio for interactive demo
!pip install -q trl==1.12.0 transformers==4.57.6 peft==0.20.0 accelerate==1.14.0 datasets==4.8.5 gradio
# Colab's torchao 0.10 breaks `import peft` 0.20; remove it
!pip uninstall -y -q torchao
!git clone https://github.com/TanWaiKen/RL-workshop.git
%cd RL-workshop/

fatal: destination path 'RL-workshop' already exists and is not an empty directory.
/content/RL-workshop


In [ ]:
%ls

README.md  relationship_env.py  tool_call_rl.ipynb


## Step 2: Check Environment
- **Checks:** Confirm installed library versions, CUDA availability, and active GPU name.


In [ ]:
import torch, trl, transformers, peft
print("trl", trl.__version__, "| transformers", transformers.__version__, "| peft", peft.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("No GPU: select Runtime > Change runtime type > T4 GPU, then rerun.")


trl 1.12.0 | transformers 4.57.6 | peft 0.20.0
CUDA available: True
GPU: Tesla T4


## Step 3: Setup Constants, Seed & Fallback Model
- **Model:** `Qwen2.5-1.5B-Instruct` from Hugging Face. If Hugging Face fails, it falls back to `Qwen2.5-1.5B-Instruct.zip` on Google Drive (set `GDRIVE_ZIP_URL` to the share link).
- **Two Scenario Sets:** Relationship Simulator train set (`N_TRAIN = 240`) and held-out eval set (`N_EVAL = 40` in fast mode, `90` in full mode).
- **Fast Mode:** `FAST_MODE = True` (default) trains SFT for 30 steps and each RL method for 20, so a full run fits a live session. Set `False` for the full run (SFT 80, RL 50).
- **Precision:** `bf16` on L4/A100, `fp16` on T4 (a T4 only emulates bf16, which is several times slower).
- **Comparison Flags:** `RUN_DPO = True`, `RUN_RLOO = True` (set False to skip a comparison stage).


In [ ]:
import os, random, zipfile
from pathlib import Path
import numpy as np, torch
from huggingface_hub import get_safetensors_metadata, snapshot_download

# -----------------------------------------------------------------------------
# Base model: Hugging Face first, Google Drive zip as fallback
# -----------------------------------------------------------------------------
HUB_MODEL_ID = "Qwen/Qwen2.5-1.5B-Instruct"
LOCAL_MODEL_DIR = "./Qwen2.5-1.5B-Instruct"
FALLBACK_ZIP_NAME = "Qwen2.5-1.5B-Instruct.zip"
# Share link of Qwen2.5-1.5B-Instruct.zip on Google Drive ("Anyone with the link").
# Used only when Hugging Face is down or rate-limited.
GDRIVE_ZIP_URL = "https://drive.google.com/file/d/1sO0ZzO0iWTuKo2aYJIOnPh7cXE9u35nt/view?usp=sharing"
# If you mounted your own Drive instead, the zip is looked up here.
GDRIVE_MOUNTED_ZIP = f"/content/drive/MyDrive/{FALLBACK_ZIP_NAME}"

def has_model(path):
    return (Path(path) / "config.json").is_file()

def fetch_fallback_zip():
    """Return a local path to the fallback zip, downloading it from Google Drive if needed."""
    if os.path.exists(FALLBACK_ZIP_NAME):
        return FALLBACK_ZIP_NAME
    if os.path.exists(GDRIVE_MOUNTED_ZIP):
        return GDRIVE_MOUNTED_ZIP
    if GDRIVE_ZIP_URL:
        import gdown
        print(f"Downloading {FALLBACK_ZIP_NAME} from Google Drive...")
        return gdown.download(GDRIVE_ZIP_URL, FALLBACK_ZIP_NAME, quiet=False, fuzzy=True)
    return None

def resolve_base_model():
    if has_model(LOCAL_MODEL_DIR):
        return LOCAL_MODEL_DIR
    try:
        return snapshot_download(HUB_MODEL_ID)
    except Exception as exc:
        print(f"Hugging Face download failed ({type(exc).__name__}: {exc}); trying Google Drive fallback.")
    zip_path = fetch_fallback_zip()
    if not zip_path:
        raise RuntimeError(
            f"Hugging Face is unavailable and no {FALLBACK_ZIP_NAME} was found. "
            "Set GDRIVE_ZIP_URL to the Drive share link, or mount Drive with the zip in MyDrive.")
    print(f"Extracting {zip_path} -> {LOCAL_MODEL_DIR}...")
    with zipfile.ZipFile(zip_path) as zf:
        zf.extractall(LOCAL_MODEL_DIR)
    nested = [p for p in Path(LOCAL_MODEL_DIR).iterdir() if p.is_dir() and has_model(p)]
    return LOCAL_MODEL_DIR if has_model(LOCAL_MODEL_DIR) else str(nested[0])

BASE_MODEL_ID = resolve_base_model()
print(f"Base model resolved to: {BASE_MODEL_ID}")

# -----------------------------------------------------------------------------
# Dataset & Training Constants (100% Simulator Data)
# -----------------------------------------------------------------------------
SEED = 42
# FAST_MODE = True fits a live workshop: fewer training steps and a smaller eval set.
# Set False for the full run (better scores, several times longer).
FAST_MODE = True
SFT_MAX_STEPS, RL_MAX_STEPS = (30, 20) if FAST_MODE else (80, 50)
N_TRAIN_SCENARIOS = 240
N_EVAL_SCENARIOS, N_PER_CATEGORY = (40, 40) if FAST_MODE else (90, 50)
NO_TOOL_FRACTION = 0.10
MIN_PER_CATEGORY = 20
EVAL_MAX_NEW_TOKENS, EVAL_BATCH_SIZE = 256, 16
OUTCOME_SCALE = 30
GRPO_NUM_GENERATIONS, MAX_COMPLETION_LENGTH = 4, 192
N_CANDIDATES = GRPO_NUM_GENERATIONS
RUN_DPO, RUN_RLOO = True, True  # Set False to skip a comparison stage

OUT = "./outputs"
SFT_ADAPTER_DIR, SFT_MERGED_DIR = f"{OUT}/sft_adapter", f"{OUT}/sft_merged"
GRPO_DIR, DPO_DIR, RLOO_DIR = f"{OUT}/grpo", f"{OUT}/dpo", f"{OUT}/rloo"
RESULTS_PATH, BASELINE_PATH = "./results/benchmark_results.json", "./results/baseline_results.json"
N_PREVIEW_SCENARIOS, N_SIDE_BY_SIDE = 3, 3
LORA = dict(r=16, lora_alpha=32, lora_dropout=0.05, task_type="CAUSAL_LM",
            target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"])

os.makedirs(OUT, exist_ok=True); os.makedirs("./results", exist_ok=True)

random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
# bf16 needs Ampere or newer (L4, A100). A T4 reports bf16 as "supported" but only emulates it,
# which makes training several times slower, so T4 must use fp16.
USE_BF16 = torch.cuda.is_available() and torch.cuda.get_device_capability()[0] >= 8; USE_FP16 = not USE_BF16
print("Precision:", "bf16" if USE_BF16 else "fp16")

try:
    if os.path.isdir(BASE_MODEL_ID):
        import json
        cfg_file = Path(BASE_MODEL_ID) / "config.json"
        if cfg_file.is_file():
            cfg = json.loads(cfg_file.read_text(encoding="utf-8"))
            print(f"{BASE_MODEL_ID} (local): architecture {cfg.get('architectures', ['CausalLM'])}")
    else:
        n_params = sum(get_safetensors_metadata(BASE_MODEL_ID).parameter_count.values())
        print(f"{BASE_MODEL_ID}: {n_params / 1e9:.2f}B parameters")
        if n_params > 1.6e9:
            print(f"Warning: {BASE_MODEL_ID} has {n_params / 1e9:.2f}B parameters; targets <=1.6B.")
except Exception as exc:
    print(f"Could not read parameter count of {BASE_MODEL_ID}: {exc}")


Base model resolved to: Qwen/Qwen2.5-1.5B-Instruct
Precision: bf16


/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Qwen/Qwen2.5-1.5B-Instruct: 1.54B parameters


## Step 4: Training Helpers
- `new_lora_config()`: Builds fresh LoRA config for PEFT.
- `free_gpu()`: Reclaims GPU VRAM between stages.
- `require_sft()`: Ensures SFT checkpoint exists before running RL.
- `load_sft_tokenizer()`: Loads tokenizer configured with SFT checkpoint.
- `run_stage()`: Orchestrates training, checkpointing, and GPU memory cleanup.


In [ ]:
import gc, json, time, traceback
from pathlib import Path
from peft import LoraConfig

def new_lora_config():
    return LoraConfig(**LORA)

def free_gpu():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        print(f"GPU memory allocated: {torch.cuda.memory_allocated() / 1e9:.2f} GB")

def require_sft():
    if not os.path.isdir(SFT_MERGED_DIR):
        raise FileNotFoundError(f"SFT checkpoint not found at {SFT_MERGED_DIR}. Run SFT first.")

def load_sft_tokenizer(padding_side=None):
    require_sft()
    kwargs = {} if padding_side is None else {"padding_side": padding_side}
    return AutoTokenizer.from_pretrained(BASE_MODEL_ID, **kwargs)

def run_stage(name, fn, after=None):
    start = time.time()
    trainer, out_dir = fn()
    try:
        trainer.train()
        trainer.save_model(out_dir)
        trainer.processing_class.save_pretrained(out_dir)
        if after is not None:
            after(trainer)
        print(f"{name}: saved to {out_dir} in {(time.time() - start) / 60:.1f} min")
    except torch.cuda.OutOfMemoryError as exc:
        traceback.clear_frames(exc.__traceback__)
        del trainer; free_gpu()
        raise RuntimeError(f"{name} ran out of GPU memory.") from exc
    del trainer; free_gpu()


## Step 5: Shared module (hidden helper)

- **Dataset:** none; this cell only stores the source of `relationship_env.py`.
- **Method:** the Relationship Simulator, the nine partner tools, the tool-call parser and printer, the reward function, and the BFCL-style AST matcher live in `relationship_env.py`. The next cell holds an exact copy of that file as the string `MODULE_SOURCE`, so the notebook still works when the file is not in the working directory (for example on a fresh Colab runtime).
- **Key settings:** none. The cell is long on purpose; run it and move on.
- **Output:** the `MODULE_SOURCE` string (nothing is written yet).


## Step 6: Import the shared module

- **Dataset:** none.
- **Method:** write `relationship_env.py` from `MODULE_SOURCE` only if the file is missing, then import it (reloading it if it was already imported) and pull in the simulator, tools, parser, printer, reward, and AST matcher.
- **Key settings:** none.
- **Output:** `./relationship_env.py` (only when it was missing).


In [ ]:
import importlib
from pathlib import Path

# Write the shared module only if it is missing (e.g. a fresh Colab runtime)
module_file = Path("relationship_env.py")
if not module_file.exists():
    module_file.write_bytes(MODULE_SOURCE.encode("utf-8"))

import relationship_env
relationship_env = importlib.reload(relationship_env)  # pick up a fresh copy

from relationship_env import (
    RelationshipSim, PARTNER_TOOLS, tool_schemas,  # Relationship_Sim + Partner_Tools
    parse_tool_calls, print_tool_calls,            # Tool_Call_Parser + Tool_Call_Printer
    tool_reward,                                   # Tool_Reward
    ast_match,                                     # AST_Matcher
)
print("relationship_env loaded:", len(PARTNER_TOOLS), "partner tools")


relationship_env loaded: 9 partner tools


## Step 7: Build Two Sets of Relation Scenarios (Train vs Eval)
- **Set 1 (Train):** 240 scenarios used to build SFT demonstrations, GRPO/RLOO prompts, and DPO chosen/rejected pairs.
- **Set 2 (Eval):** 90 held-out scenarios used to build the BFCL evaluation suite (never seen during training).
- **100% Simulator Data:** No external datasets; eliminates model cheating concerns.
- sft (Supervised Fine-Tuning) — 240 Examples with correct answer
- rl (Reinforcement Learning / GRPO & RLOO) — 240 Examples. Unlike SFT, there are no correct answers provided in this split.
- dpo (Direct Preference Optimization) — 720 Pairs (240x3) (chosen vs. rejected answers)


In [ ]:
from transformers import AutoTokenizer
from relationship_env import RelationshipSim

SFT_MAX_LENGTH = 2048
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_ID)
sim = RelationshipSim(seed=SEED)

# Generate Set 1 (Train) and Set 2 (Held-out Eval, guaranteed disjoint)
train_scenarios = sim.generate_scenarios(N_TRAIN_SCENARIOS, "train")
eval_scenarios = sim.generate_scenarios(
    N_EVAL_SCENARIOS, "eval", exclude={s.scenario_id for s in train_scenarios})

print(f"Built Two Scenario Sets: {len(train_scenarios)} train scenarios | {len(eval_scenarios)} eval scenarios")


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Built Two Scenario Sets: 240 train scenarios | 90 eval scenarios


In [ ]:
from relationship_env import (
    build_sft_examples, build_rl_examples, build_preference_pairs, build_bfcl_eval,
    make_dataset, check_template_has_tools, check_min_per_category, print_data_summary)

sft_rows = build_sft_examples(train_scenarios, sim, rng=random.Random(SEED))
rl_rows = build_rl_examples(train_scenarios, tokenizer)
dpo_rows, dpo_summary = build_preference_pairs(
    train_scenarios, sim, N_CANDIDATES, random.Random(f"{SEED}-dpo"))
eval_items = build_bfcl_eval(sim, eval_scenarios, n_per_category=N_PER_CATEGORY)

# Verify tool names survive chat template
for row in sft_rows + dpo_rows[:1]:
    text = tokenizer.apply_chat_template(row.get("messages") or row["prompt"], tools=row["tools"], tokenize=False)
    check_template_has_tools(text, row["tools"])

check_min_per_category(eval_items, MIN_PER_CATEGORY)
sft_dataset, rl_dataset, dpo_dataset = (make_dataset(rows) for rows in (sft_rows, rl_rows, dpo_rows))

print_data_summary(
    {"sft": len(sft_rows), "rl": len(rl_rows), "dpo": len(dpo_rows),
     "eval": sum(len(v) for k, v in eval_items.items() if k != "sim")},
    eval_items, {"train": train_scenarios, "eval": eval_scenarios}, dpo_summary=dpo_summary)


Data summary
Examples per split:
  sft          240
  rl           240
  dpo          720
  eval         100
Eval examples per category (sim + public):
  simple       25 (25 sim + 0 public)
  multiple     25 (25 sim + 0 public)
  parallel     25 (25 sim + 0 public)
  irrelevance  25 (25 sim + 0 public)
  sim          90 (90 sim + 0 public)
Scenarios per occasion:
  occasion           train    eval
  after_argument        40      15
  anniversary           40      15
  bad_day_at_work       40      15
  birthday              40      15
  normal_evening        40      15
  rainy_weekend         40      15
  (no_tool)             24       9
Dropped public examples per reason:
  malformed_json         0
  arguments_not_object   0
  unknown_tool           0
  too_long               0
DPO pairs dropped for tied scores: 0


'Data summary\nExamples per split:\n  sft          240\n  rl           240\n  dpo          720\n  eval         100\nEval examples per category (sim + public):\n  simple       25 (25 sim + 0 public)\n  multiple     25 (25 sim + 0 public)\n  parallel     25 (25 sim + 0 public)\n  irrelevance  25 (25 sim + 0 public)\n  sim          90 (90 sim + 0 public)\nScenarios per occasion:\n  occasion           train    eval\n  after_argument        40      15\n  anniversary           40      15\n  bad_day_at_work       40      15\n  birthday              40      15\n  normal_evening        40      15\n  rainy_weekend         40      15\n  (no_tool)             24       9\nDropped public examples per reason:\n  malformed_json         0\n  arguments_not_object   0\n  unknown_tool           0\n  too_long               0\nDPO pairs dropped for tied scores: 0'

### Step 7b: Preview Data and Rewards
- Inspects partner state, user prompt, and optimal tool-call plan.
- Shows rule-based reward breakdown (`format`, `syntax`, `outcome`, `exact`).


In [ ]:
from relationship_env import OUTCOME_SCALE as SIM_OUTCOME_SCALE

for i, s in enumerate([s for s in train_scenarios if not s.is_no_tool][:N_PREVIEW_SCENARIOS], 1):
    st, p = s.state, s.state.preferences
    print(f"=== Scenario {i} ===")
    print(f"Partner: mood={st.mood}, love_meter={st.love_meter}, occasion={st.occasion}")
    print(f"Preferences: gifts={', '.join(p.favorite_gift_types)} | likes={', '.join(p.liked_genres)} | "
          f"dislikes={', '.join(p.disliked_genres)} | food={p.favorite_food} | love language={p.love_language}")
    print("Request:", s.request)
    print(f"Optimal plan ({len(s.optimal_plan)} turn(s), love {s.optimal_delta:+d}):")
    print(print_tool_calls([call for group in s.optimal_plan for call in group]), end="\n\n")

REWARD_ROWS = [
    ("format", "0 or 0.5", "answer starts with one non-empty <think>...</think> block"),
    ("syntax", "0 or 0.5", "every <tool_call> block is valid JSON naming an offered tool"),
    ("outcome", "-1 to 1", f"Love_Delta / {SIM_OUTCOME_SCALE}, clipped"),
    ("exact", "0 or 1", "calls match the Optimal_Plan exactly"),
]
print(f"{'component':<10} {'range':<9} condition")
for name, rng, cond in REWARD_ROWS:
    print(f"{name:<10} {rng:<9} {cond}")


=== Scenario 1 ===
Partner: mood=angry, love_meter=60, occasion=after_argument
Preferences: gifts=jewelry | likes=documentary | dislikes=comedy | food=pasta | love language=gifts
Request: I said something thoughtless to Ethan today. Can you help me make it up to him?
Optimal plan (2 turn(s), love +40):
<tool_call>
{"name": "apologize", "arguments": {"reason": "I love you"}}
</tool_call>
<tool_call>
{"name": "give_gift", "arguments": {"item": "necklace"}}
</tool_call>

=== Scenario 2 ===
Partner: mood=happy, love_meter=17, occasion=anniversary
Preferences: gifts=book | likes=comedy | dislikes=horror | food=tacos | love language=quality_time
Request: I want to celebrate our anniversary with Grace. Any ideas?
Optimal plan (1 turn(s), love +55):
<tool_call>
{"name": "give_gift", "arguments": {"item": "novel"}}
</tool_call>
<tool_call>
{"name": "give_gift", "arguments": {"item": "poetry_book"}}
</tool_call>
<tool_call>
{"name": "plan_date", "arguments": {"activity": "beach_walk"}}
</tool_ca

## Step 8: Benchmark Helpers
- Evaluates models using greedy decoding on held-out BFCL test categories.
- Tracks tool-call accuracy, AST syntax validity, and love meter score across Base, SFT, GRPO, DPO, and RLOO.


In [ ]:
import pandas as pd
from transformers import AutoModelForCausalLM
from peft import PeftModel
from relationship_env import eval_prompt_texts, score_eval_outputs, benchmark_results, format_headline_row, write_results_file

EVAL_MODELS = [
    ("base", BASE_MODEL_ID, None),
    ("sft", SFT_MERGED_DIR, None),
    ("grpo", SFT_MERGED_DIR, GRPO_DIR),
    ("dpo", SFT_MERGED_DIR, DPO_DIR),
    ("rloo", SFT_MERGED_DIR, RLOO_DIR),
]
DECODING = {"do_sample": False, "max_new_tokens": EVAL_MAX_NEW_TOKENS, "batch_size": EVAL_BATCH_SIZE}

def load_eval_model(path, adapter=None):
    tok = (load_sft_tokenizer(padding_side="left") if path == SFT_MERGED_DIR
           else AutoTokenizer.from_pretrained(path, padding_side="left"))
    tok.pad_token = tok.pad_token or tok.eos_token
    model = AutoModelForCausalLM.from_pretrained(path, dtype=torch.bfloat16 if USE_BF16 else torch.float16, device_map="auto")
    return (PeftModel.from_pretrained(model, adapter) if adapter else model).eval(), tok

@torch.no_grad()
def generate_batch(model, tokenizer, prompts):
    outputs = []
    for i in range(0, len(prompts), EVAL_BATCH_SIZE):
        enc = tokenizer(prompts[i:i + EVAL_BATCH_SIZE], return_tensors="pt", padding=True, add_special_tokens=False).to(model.device)
        gen = model.generate(**enc, do_sample=False, max_new_tokens=EVAL_MAX_NEW_TOKENS, pad_token_id=tokenizer.pad_token_id)
        outputs += tokenizer.batch_decode(gen[:, enc["input_ids"].shape[1]:], skip_special_tokens=True)
    return outputs

def run_benchmark(models, eval_items):
    scores, skipped = {}, []
    for name, path, adapter in models:
        missing = [p for p in (path, adapter) if p and p.startswith(OUT) and not os.path.isdir(p)]
        if missing:
            print(f"Skipping {name}: {missing[0]} not found"); skipped.append(name); continue
        model, tok = load_eval_model(path, adapter)
        outputs = {c: generate_batch(model, tok, eval_prompt_texts(items, tok)) for c, items in eval_items.items()}
        scores[name] = score_eval_outputs(name, eval_items, outputs, sim)
        del model; free_gpu()
    return scores, skipped

headline_table = lambda scores: pd.DataFrame([format_headline_row(s["headline"]) for s in scores.values()], index=list(scores))


## Step 9: Baseline Benchmark (Untrained Base Model)
- Evaluates `Qwen2.5-1.5B-Instruct` prior to fine-tuning on the held-out eval scenario set.


In [ ]:
baseline_scores, _ = run_benchmark(EVAL_MODELS[:1], eval_items)
display(headline_table(baseline_scores))
write_results_file(BASELINE_PATH, benchmark_results(baseline_scores, eval_items, BASE_MODEL_ID, SEED, DECODING))
base_stats = baseline_scores["base"]["stats"]
print(f"Baseline saved to {BASELINE_PATH}")


GPU memory allocated: 3.20 GB


,BFCL AST Acc,Irrelevance Acc,Sim Task Success,Mean Love Gain
base,5.3%,68.0%,2.2%,2.1


Baseline saved to ./results/baseline_results.json


## Step 10: SFT Warm-up
- **Goal:** Teaches the model the basic JSON `<tool_call>` syntax via demonstration on Set 1 (Train).
- **Merge:** Merges adapter into base weights so GRPO, DPO, and RLOO all branch from this shared foundation.


In [ ]:
from trl import SFTConfig, SFTTrainer

DTYPE = torch.bfloat16 if USE_BF16 else torch.float16

def build_sft():
    args = SFTConfig(
        output_dir=SFT_ADAPTER_DIR,
        learning_rate=2e-4, max_steps=SFT_MAX_STEPS,
        per_device_train_batch_size=2, gradient_accumulation_steps=4,
        max_length=SFT_MAX_LENGTH,
        logging_steps=5,
        bf16=USE_BF16, fp16=USE_FP16, report_to="none", seed=SEED)
    model = AutoModelForCausalLM.from_pretrained(BASE_MODEL_ID, dtype=DTYPE)
    trainer = SFTTrainer(model=model, args=args, train_dataset=sft_dataset,
                         processing_class=tokenizer, peft_config=new_lora_config())
    return trainer, SFT_ADAPTER_DIR

def merge_sft(trainer):
    base = AutoModelForCausalLM.from_pretrained(BASE_MODEL_ID, dtype=DTYPE)
    merged = PeftModel.from_pretrained(base, SFT_ADAPTER_DIR).merge_and_unload()
    merged.save_pretrained(SFT_MERGED_DIR); tokenizer.save_pretrained(SFT_MERGED_DIR)
    print(f"SFT checkpoint (merged) saved to {SFT_MERGED_DIR}")
    del base, merged

run_stage("SFT", build_sft, after=merge_sft)


## Step 11: GRPO (Primary Tool-Calling RL)
- **Method:** Group Relative Policy Optimization (4 completions per prompt).
- **Reward:** Rule-based reward on `<think>`, tool syntax, and love meter delta.
- **Budget:** `RL_MAX_STEPS` steps (20 in fast mode, 50 in full), learning rate 2e-5.


In [ ]:
from trl import GRPOConfig, GRPOTrainer

def build_grpo():
    require_sft()
    args = GRPOConfig(
        output_dir=GRPO_DIR,
        learning_rate=2e-5, max_steps=RL_MAX_STEPS,
        per_device_train_batch_size=8, gradient_accumulation_steps=1,
        num_generations=GRPO_NUM_GENERATIONS,
        max_completion_length=MAX_COMPLETION_LENGTH,
        beta=0.0,
        logging_steps=5,
        bf16=USE_BF16, fp16=USE_FP16, report_to="none", seed=SEED)
    tok = load_sft_tokenizer(padding_side="left")
    tok.pad_token = tok.pad_token or tok.eos_token
    model = AutoModelForCausalLM.from_pretrained(
        SFT_MERGED_DIR, dtype=torch.bfloat16 if USE_BF16 else torch.float16)
    trainer = GRPOTrainer(model=model, reward_funcs=tool_reward, args=args, train_dataset=rl_dataset,
                          processing_class=tok, peft_config=new_lora_config())
    return trainer, GRPO_DIR

run_stage("GRPO", build_grpo)


## Step 12: DPO (Comparison Method)
- **Method:** Direct Preference Optimization trains on chosen (optimal) vs rejected (perturbed) tool-call pairs.
- **Reference:** References the SFT policy without generating completions during training.


In [ ]:
from trl import DPOConfig, DPOTrainer

def build_dpo():
    require_sft()
    args = DPOConfig(
        output_dir=DPO_DIR,
        learning_rate=5e-5, max_steps=RL_MAX_STEPS,
        per_device_train_batch_size=2, gradient_accumulation_steps=4,
        max_length=SFT_MAX_LENGTH,
        beta=0.1,
        logging_steps=5,
        bf16=USE_BF16, fp16=USE_FP16, report_to="none", seed=SEED)
    tok = AutoTokenizer.from_pretrained(SFT_MERGED_DIR)
    tok.pad_token = tok.pad_token or tok.eos_token
    model = AutoModelForCausalLM.from_pretrained(
        SFT_MERGED_DIR, dtype=torch.bfloat16 if USE_BF16 else torch.float16)
    trainer = DPOTrainer(model=model, ref_model=None, args=args, train_dataset=dpo_dataset,
                         processing_class=tok, peft_config=new_lora_config())
    return trainer, DPO_DIR

if RUN_DPO:
    run_stage("DPO", build_dpo)
else:
    print("DPO skipped (RUN_DPO = False)")


## Step 13: RLOO (Comparison Method)
- **Method:** Reinforce Leave-One-Out uses sample-mean of sibling completions as baseline.
- **Budget:** Same 4 completions per prompt and same `RL_MAX_STEPS` as GRPO for a fair comparison.


In [ ]:
from trl import RLOOConfig, RLOOTrainer

def build_rloo():
    require_sft()
    args = RLOOConfig(
        output_dir=RLOO_DIR,
        learning_rate=2e-5, max_steps=RL_MAX_STEPS,
        per_device_train_batch_size=8, gradient_accumulation_steps=1,
        num_generations=GRPO_NUM_GENERATIONS,
        max_completion_length=MAX_COMPLETION_LENGTH,
        beta=0.05,
        logging_steps=5,
        bf16=USE_BF16, fp16=USE_FP16, report_to="none", seed=SEED)
    tok = load_sft_tokenizer(padding_side="left")
    tok.pad_token = tok.pad_token or tok.eos_token
    model = AutoModelForCausalLM.from_pretrained(
        SFT_MERGED_DIR, dtype=torch.bfloat16 if USE_BF16 else torch.float16)
    trainer = RLOOTrainer(model=model, reward_funcs=tool_reward, args=args, train_dataset=rl_dataset,
                          processing_class=tok, peft_config=new_lora_config())
    return trainer, RLOO_DIR

if RUN_RLOO:
    run_stage("RLOO", build_rloo)
else:
    print("RLOO skipped (RUN_RLOO = False)")


## How to Read the Benchmark Scores
- **BFCL AST Acc:** Syntactic accuracy (correct tool name and schema-compliant JSON arguments).
- **Sim Task Success:** Complete plan match on the Relationship Simulator.
- **Sim Mean Love Delta:** Average love points gained per interaction.
- **Mean Reward:** Overall score including format (`<think>`) and outcome components.


## Step 14: Final Benchmark (Comparing All Methods)
- Evaluates Base, SFT, GRPO, DPO, and RLOO on Set 2 (held-out eval scenarios).
- Outputs headline metrics, delta change vs base, and per-category accuracy.


In [ ]:
from relationship_env import load_baseline_scores, delta_rows, format_pct

skipped_stages = [name for name, run in (("dpo", RUN_DPO), ("rloo", RUN_RLOO)) if not run]
for name in skipped_stages:
    print(f"Skipping {name}: stage turned off")

base_scores = load_baseline_scores(BASELINE_PATH, eval_items, BASE_MODEL_ID, SEED, DECODING)
to_score = [m for m in EVAL_MODELS if m[0] not in skipped_stages and not (m[0] == "base" and base_scores)]
final_scores, missing = run_benchmark(to_score, eval_items)
if base_scores:
    final_scores = {"base": base_scores, **final_scores}

trained = [m for m in final_scores if m != "base"]
display(headline_table(final_scores))
if trained:
    display(pd.DataFrame.from_dict(delta_rows(final_scores), orient="index"))
display(pd.DataFrame({m: {c: format_pct(v) for c, v in s["bfcl_per_category"].items()}
                      for m, s in final_scores.items()}).T)

write_results_file(RESULTS_PATH, benchmark_results(
    final_scores, eval_items, BASE_MODEL_ID, SEED, DECODING, skipped_stages + missing))
print(f"Results saved to {RESULTS_PATH}")


### Step 14b: Side-by-Side Scenarios
- Compares tool choices and love meter outcomes of the Base model vs the best trained model.


In [ ]:
from relationship_env import side_by_side_rows, best_trained_model

best = best_trained_model(final_scores)
if best:
    print(f"Best trained model: {best}")
    with pd.option_context("display.max_colwidth", None):
        display(pd.DataFrame(side_by_side_rows(final_scores, eval_items, best, n=N_SIDE_BY_SIDE)))


### Step 14c: Reward-Hacking Check
- Confirms the model improves real outcomes rather than exploiting reward loopholes.


In [ ]:
from relationship_env import reward_hacking_flags, format_hacking_flags

best_model = best_trained_model(final_scores) or "grpo"
if best_model in final_scores and "base" in final_scores:
    flags = reward_hacking_flags(final_scores[best_model]["stats"], final_scores["base"]["stats"])
    print(f"Reward-hacking check for {best_model}:")
    for line in format_hacking_flags(flags):
        print(" ", line)


## Step 15: Interactive Demo — Chat with Your Model (Gradio)
- **In-Notebook:** Runs natively inside Google Colab or local Jupyter via Gradio (`inline=True`).
- **Model Selector:** Switch between trained models (`grpo`, `dpo`, `rloo`, `sft`, or `base`).
- **Live Tool Execution:** Model reasons inside `<think>`, executes tools, and updates the love meter live.
- **Controls:** Type messages or click **🎲 New Scenario** to test different relationship situations.


In [ ]:
import os, json, random
from pathlib import Path
import gradio as gr
from relationship_env import (
    Episode, RelationshipSim, Scenario, parse_tool_calls, system_message, tool_schemas
)

class NotebookChatSession:
    def __init__(self, default_path="./outputs/grpo"):
        self.model_path = default_path
        self.model = None
        self.tokenizer = None
        self.device = "cuda" if torch.cuda.is_available() else "cpu"
        self.sim = RelationshipSim(seed=random.randrange(2**31))
        self.scenario = self.sim.generate_scenarios(1, "eval")[0]
        self.episode = Episode(self.scenario)
        self.messages = [{"role": "system", "content": system_message(self.scenario)}]

    def get_status_text(self):
        st = self.episode.state
        p = st.preferences
        return (
            f"### Partner: {self.scenario.partner_name}\n"
            f"- **Mood:** `{st.mood}` | **Occasion:** `{st.occasion.replace('_', ' ')}`\n"
            f"- **Love Meter:** **{self.episode.love_meter} / 100**\n"
            f"- **Preferences:** Gifts: {', '.join(p.favorite_gift_types)} | Likes: {', '.join(p.liked_genres)} | Dislikes: {', '.join(p.disliked_genres)}\n"
            f"- **Suggested Prompt:** *\"{self.scenario.request}\"*\n"
        )

    def reset_scenario(self):
        self.scenario = self.sim.generate_scenarios(1, "eval")[0]
        self.episode = Episode(self.scenario)
        self.messages = [{"role": "system", "content": system_message(self.scenario)}]
        return self.get_status_text()

    def load_model(self, path):
        dtype = torch.bfloat16 if USE_BF16 else (torch.float16 if self.device == "cuda" else torch.float32)
        target = path.strip() if path and path.strip() else "./outputs/grpo"
        if not os.path.exists(target):
            target = BASE_MODEL_ID
        root = Path(target)
        adapter_cfg = root / "adapter_config.json"
        if adapter_cfg.is_file():
            from peft import PeftModel
            base_id = json.loads(adapter_cfg.read_text(encoding="utf-8")).get("base_model_name_or_path", BASE_MODEL_ID)
            base = AutoModelForCausalLM.from_pretrained(base_id, dtype=dtype)
            self.model = PeftModel.from_pretrained(base, target)
            tok_path = target if (root / "tokenizer_config.json").is_file() else base_id
        else:
            self.model = AutoModelForCausalLM.from_pretrained(target, dtype=dtype)
            tok_path = target
        self.tokenizer = AutoTokenizer.from_pretrained(tok_path)
        self.model.to(self.device)
        self.model.eval()
        self.model_path = target
        return f"Loaded `{target}` on {self.device.upper()}"

    def chat_turn(self, user_msg, history):
        if not user_msg.strip():
            return history, self.get_status_text()
        if self.model is None:
            self.load_model(self.model_path)
        self.episode.begin_turn()
        self.messages.append({"role": "user", "content": user_msg})
        history = history or []
        history.append({"role": "user", "content": user_msg})

        tools = tool_schemas()
        pad_id = self.tokenizer.pad_token_id or self.tokenizer.eos_token_id
        parts = []

        for _ in range(5):
            prompt = self.tokenizer.apply_chat_template(self.messages, tools=tools, tokenize=False, add_generation_prompt=True)
            enc = self.tokenizer(prompt, return_tensors="pt").to(self.device)
            with torch.no_grad():
                gen = self.model.generate(**enc, do_sample=True, temperature=0.7, max_new_tokens=256, pad_token_id=pad_id)
            raw = self.tokenizer.decode(gen[0, enc["input_ids"].shape[1]:], skip_special_tokens=True)
            self.messages.append({"role": "assistant", "content": raw})
            parsed = parse_tool_calls(raw)
            if parsed.think:
                parts.append(f"💭 **Reasoning:**\n> {parsed.think.strip()}")
            if not parsed.calls and not parsed.errors:
                if parsed.text:
                    parts.append(parsed.text.strip())
                break
            for call in parsed.calls:
                res = self.episode.execute(call)
                self.messages.append({"role": "tool", "name": call.name, "content": json.dumps(res, ensure_ascii=False)})
                delta = res.get("love_delta", 0)
                sign = f"+{delta}" if delta > 0 else str(delta)
                parts.append(f"🔧 **Tool Call:** `{call.name}({json.dumps(dict(call.arguments))})`\n"
                             f"→ Outcome: *{res.get('result', '')}* (Love: **{sign}**, Meter: **{res.get('love_meter')}/100**)")
            for err in parsed.errors:
                self.messages.append({"role": "tool", "name": "error", "content": json.dumps({"error": err.reason})})
                parts.append(f"⚠️ **Tool Error:** {err.reason}")

        history.append({"role": "assistant", "content": "\n\n".join(parts) or "(no response)"})
        return history, self.get_status_text()

# Launch in-notebook Gradio demo
chat_session = NotebookChatSession()

with gr.Blocks(title="Relationship Simulator") as demo:
    gr.Markdown("## 💞 Relationship Simulator — Interactive Tool-Calling Demo")
    gr.Markdown("Directly chat with your trained model. The model reasons inside `<think>`, calls partner tools, and updates love meter live.")
    with gr.Row():
        with gr.Column(scale=1):
            status_box = gr.Markdown(chat_session.get_status_text())
            new_btn = gr.Button("🎲 New Scenario", variant="secondary")
            gr.Markdown("---")
            presets = ["./outputs/grpo", "./outputs/dpo", "./outputs/rloo", "./outputs/sft_merged", BASE_MODEL_ID]
            model_drop = gr.Dropdown(choices=presets, value=presets[0], label="Model Presets", allow_custom_value=True)
            load_btn = gr.Button("🔄 Load Model")
            load_lbl = gr.Markdown("Click Load Model to start.")
        with gr.Column(scale=2):
            chatbot = gr.Chatbot(type="messages", height=450)
            with gr.Row():
                msg_in = gr.Textbox(placeholder="Say something to your partner...", label="Your Message", scale=4)
                send_btn = gr.Button("Send", variant="primary", scale=1)

    def on_send(msg, hist):
        hist, st_text = chat_session.chat_turn(msg, hist)
        return "", hist, st_text

    send_btn.click(on_send, [msg_in, chatbot], [msg_in, chatbot, status_box])
    msg_in.submit(on_send, [msg_in, chatbot], [msg_in, chatbot, status_box])
    new_btn.click(lambda: ([], chat_session.reset_scenario()), outputs=[chatbot, status_box])
    load_btn.click(lambda p: chat_session.load_model(p), [model_drop], [load_lbl])

demo.launch(inline=True, share=True)
